# Hierarchical ReLU-LoRA on Qwen1.5-MoE

This notebook implements the Hierarchical ReLU-LoRA method on `Qwen/Qwen1.5-MoE-A2.7B-Chat`.

**Changes from Phi-MoE version:**
- Model: Qwen1.5-MoE-A2.7B-Chat (working MoE model)
- Expert wrapper adapted for Qwen's architecture
- Prompt format updated for Qwen chat template

In [ ]:
# ── Setup save directory (Colab or Kaggle) ────────────────────────────────────
import os

# Auto-detect platform and set SAVE_ROOT accordingly
if os.path.exists('/kaggle'):
    SAVE_ROOT = "/kaggle/working/hrlora_checkpoints"
    print("Detected: Kaggle")
elif os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/hrlora_checkpoints"
        print("Detected: Colab (Drive mounted)")
    except:
        SAVE_ROOT = "/content/hrlora_checkpoints"
        print("Detected: Colab (Drive not available)")
else:
    SAVE_ROOT = "./hrlora_checkpoints"
    print("Detected: Local environment")

os.makedirs(SAVE_ROOT, exist_ok=True)
print(f"Saving to: {SAVE_ROOT}")

In [ ]:
# ── Load Qwen1.5-MoE Model ────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "Qwen/Qwen1.5-MoE-A2.7B-Chat"

print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model in bfloat16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.eval()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"Model type: {type(model).__name__}")

In [ ]:
# ── MODEL LOAD TEST: Verify model works before proceeding ────────────────────
print("="*60)
print("MODEL LOAD TEST")
print("="*60)

# Check config
print(f"\nModel Config:")
print(f"  vocab_size: {model.config.vocab_size}")
print(f"  hidden_size: {model.config.hidden_size}")
print(f"  num_layers: {model.config.num_hidden_layers}")
print(f"  num_experts: {getattr(model.config, 'num_experts', 'N/A')}")
print(f"  num_experts_per_tok: {getattr(model.config, 'num_experts_per_tok', 'N/A')}")

# Check tokenizer match
print(f"\nTokenizer:")
print(f"  vocab_size: {len(tokenizer)}")
print(f"  Model expects: {model.config.vocab_size}")
print(f"  Match: {len(tokenizer) == model.config.vocab_size or len(tokenizer) <= model.config.vocab_size}")

# Test generation
print(f"\n--- Simple Generation Test ---")
device = next(model.parameters()).device

test_prompt = "Write a Python function to add two numbers:\n```python\ndef add(a, b):\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(
        inputs.input_ids,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"Completion: {completion}")

# Check if output looks reasonable
is_garbage = any([
    completion.count(')') > 20,  # Too many closing parens
    len(set(completion.split())) < 3,  # Too repetitive
    'return' not in completion.lower() and '+' not in completion  # No code-like content
])

print(f"\n{'='*60}")
if is_garbage:
    print("WARNING: Output looks like garbage. Model may not be working correctly.")
else:
    print("SUCCESS: Model generates reasonable output. Ready to proceed!")
print(f"{'='*60}")

In [ ]:
# ── Explore Qwen MoE Architecture ─────────────────────────────────────────────
# We need to understand where the experts are located in Qwen's architecture

print("Exploring Qwen MoE Architecture...")
print("="*60)

# Check first layer structure
layer0 = model.model.layers[0]
print(f"\nLayer 0 type: {type(layer0).__name__}")
print(f"Layer 0 children:")
for name, child in layer0.named_children():
    print(f"  {name}: {type(child).__name__}")

# Check MLP/MoE structure
if hasattr(layer0, 'mlp'):
    mlp = layer0.mlp
    print(f"\nMLP type: {type(mlp).__name__}")
    print(f"MLP children:")
    for name, child in mlp.named_children():
        print(f"  {name}: {type(child).__name__}")
        if 'expert' in name.lower():
            print(f"    -> This is likely the experts module!")
            if hasattr(child, '__len__'):
                print(f"    -> Number of experts: {len(child)}")

# Look for gate/router
print(f"\nSearching for router/gate...")
for name, module in mlp.named_modules():
    if 'gate' in name.lower() or 'router' in name.lower():
        print(f"  Found: {name} -> {type(module).__name__}")

In [ ]:
# ── HierarchicalExpert: Low-rank LoRA sub-adapter ────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter. Computes: L(x) = (x @ A.T @ B.T) * scaling
    
    B is zero-initialized so at spawn time, contribution is exactly zero.
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = base_rank
        self.scaling = lora_alpha / base_rank
        
        # Standard LoRA init: A ~ N(0, 0.01), B = 0
        self.A = nn.Parameter(torch.randn(base_rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast to match input dtype
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.T @ B.T) * self.scaling

print("HierarchicalExpert defined.")

In [ ]:
# ── HierarchicalMoEWrapper: Wrapper for Qwen MoE experts ─────────────────────
# This needs to be adapted based on the architecture exploration above

class HierarchicalMoEWrapper(nn.Module):
    """
    Wrapper for Qwen MoE experts that adds hierarchical LoRA adapters.
    
    This wraps the original MoE module and adds per-expert LoRA corrections:
        output = original_output + sum_k(lora_k(x) * routing_weight_k)
    """
    
    def __init__(self, original_moe, base_rank=16, lora_alpha=32):
        super().__init__()
        self.original = original_moe
        
        # Detect number of experts and dimensions
        self.num_experts = self._detect_num_experts()
        self.hidden_size = self._detect_hidden_size()
        
        print(f"  Detected {self.num_experts} experts, hidden_size={self.hidden_size}")
        
        # Get device and dtype from original module
        param = next(self.original.parameters())
        dev, dtype = param.device, param.dtype
        
        # Create base LoRA adapters for each expert
        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.hidden_size,
                out_features=self.hidden_size,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(dtype)
            for _ in range(self.num_experts)
        ])
        
        # Spawned sub-adapters (added dynamically)
        self.spawn_loras: list = [[] for _ in range(self.num_experts)]
        self.spawn_gates: list = [[] for _ in range(self.num_experts)]
        
        # Freeze original module
        for p in self.original.parameters():
            p.requires_grad = False
    
    def _detect_num_experts(self):
        """Detect number of experts from original module."""
        if hasattr(self.original, 'experts'):
            if hasattr(self.original.experts, '__len__'):
                return len(self.original.experts)
        # Try to get from config
        for name, module in self.original.named_modules():
            if hasattr(module, 'weight') and 'expert' in name:
                # Count unique expert indices
                pass
        # Default fallback
        return 64  # Qwen MoE default
    
    def _detect_hidden_size(self):
        """Detect hidden size from original module."""
        for name, param in self.original.named_parameters():
            if 'weight' in name:
                # Usually the last dimension is hidden_size for input projections
                return param.shape[-1]
        return 2048  # Qwen default
    
    def spawn(self, expert_id: int, rank: int = 8, weight_grad=None):
        """
        Spawn a new ReLU-gated LoRA sub-adapter for the given expert.
        """
        param = next(self.original.parameters())
        dev, dtype = param.device, param.dtype
        
        # Create new sub-adapter
        lora = HierarchicalExpert(
            in_features=self.hidden_size,
            out_features=self.hidden_size,
            base_rank=rank,
            lora_alpha=2 * rank,
        ).to(dev).to(dtype)
        
        # Initialize A from gradient SVD if available
        if weight_grad is not None:
            try:
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                lora.A.data = Vh[:rank, :].to(dtype)
            except:
                pass
        
        # Create ReLU gate vector
        gate = nn.Parameter(torch.randn(self.hidden_size, device=dev, dtype=dtype) * 0.01)
        
        self.spawn_loras[expert_id].append(lora)
        self.spawn_gates[expert_id].append(gate)
        
        return [*lora.parameters(), gate]
    
    def forward(self, hidden_states, *args, **kwargs):
        """
        Forward pass: original output + LoRA corrections.
        
        Note: This is a simplified version. The actual implementation
        depends on how Qwen's MoE returns routing information.
        """
        # Get original output
        # Qwen MoE might return (output, router_logits) or just output
        orig_result = self.original(hidden_states, *args, **kwargs)
        
        if isinstance(orig_result, tuple):
            orig_out, router_logits = orig_result[0], orig_result[1] if len(orig_result) > 1 else None
        else:
            orig_out = orig_result
            router_logits = None
        
        # For now, apply a simple average of all LoRA corrections
        # TODO: Weight by actual routing decisions when we understand the architecture better
        correction = torch.zeros_like(orig_out)
        
        for k in range(min(self.num_experts, len(self.base_loras))):
            # Base LoRA correction (averaged across experts)
            base_out = self.base_loras[k](hidden_states)
            correction = correction + base_out / self.num_experts
            
            # Spawned sub-adapter corrections
            for gate_vec, sub_lora in zip(self.spawn_gates[k], self.spawn_loras[k]):
                g = F.relu(hidden_states @ gate_vec)
                sub_out = sub_lora(hidden_states)
                correction = correction + sub_out * g.unsqueeze(-1) / self.num_experts
        
        result = orig_out + correction.to(orig_out.dtype)
        
        if isinstance(orig_result, tuple):
            return (result,) + orig_result[1:]
        return result

print("HierarchicalMoEWrapper defined.")

In [ ]:
# ── Test wrapper on first layer ──────────────────────────────────────────────
print("Testing wrapper on layer 0...")

# Get the MoE module from layer 0
layer0 = model.model.layers[0]
original_mlp = layer0.mlp

print(f"Original MLP type: {type(original_mlp).__name__}")

# Create wrapper
wrapper = HierarchicalMoEWrapper(original_mlp, base_rank=16, lora_alpha=32)

# Replace in model
layer0.mlp = wrapper

# Test forward pass
print("\nTesting forward pass with wrapper...")
test_input = tokenizer("Hello world", return_tensors="pt").to(device)

try:
    with torch.no_grad():
        out = model.generate(
            test_input.input_ids,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    completion = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Output with wrapper: {completion}")
    print("SUCCESS: Wrapper works!")
except Exception as e:
    print(f"ERROR: {e}")
    print("\nRestoring original MLP...")
    layer0.mlp = original_mlp
    print("Need to debug wrapper forward() method.")

In [ ]:
# ── Restore original model for further experiments ───────────────────────────
# Unwrap if wrapper was applied
layer0 = model.model.layers[0]
if hasattr(layer0.mlp, 'original'):
    layer0.mlp = layer0.mlp.original
    print("Restored original MLP.")
else:
    print("MLP was not wrapped.")

In [ ]:
# ── ConflictSaturationMonitor ─────────────────────────────────────────────────
import numpy as np

class ConflictSaturationMonitor:
    """
    Monitors training for signs of conflict saturation that trigger spawning.
    
    Triggers when:
    1. Loss plateau detected (gradient magnitude below threshold)
    2. Domain conflict detected (different loss trajectories for different domains)
    """
    
    def __init__(self, tau_plateau=1e-4, delta_threshold=0.5, window=15):
        self.tau_plateau = tau_plateau
        self.delta_threshold = delta_threshold
        self.window = window
        
        self.loss_history = []
        self.domain_losses = {'code': [], 'medical': []}
        self.grad_norms = []
    
    def update(self, lora_A, lora_B, loss_val, domain):
        """Update monitor with current step info. Returns True if spawn triggered."""
        self.loss_history.append(loss_val)
        
        if domain in self.domain_losses:
            self.domain_losses[domain].append(loss_val)
        
        # Compute gradient norm
        if lora_A.grad is not None and lora_B.grad is not None:
            grad_norm = (lora_A.grad.norm() + lora_B.grad.norm()).item()
            self.grad_norms.append(grad_norm)
        
        # Check for spawn trigger
        if len(self.loss_history) < self.window:
            return False
        
        # Plateau detection
        recent_grads = self.grad_norms[-self.window:] if self.grad_norms else [1.0]
        avg_grad = np.mean(recent_grads)
        plateau = avg_grad < self.tau_plateau
        
        # Conflict detection
        code_losses = self.domain_losses['code'][-self.window:]
        med_losses = self.domain_losses['medical'][-self.window:]
        
        if len(code_losses) >= 3 and len(med_losses) >= 3:
            delta = abs(np.mean(code_losses) - np.mean(med_losses))
            conflict = delta > self.delta_threshold
        else:
            conflict = False
        
        return plateau or conflict
    
    def reset_after_spawn(self):
        """Reset monitors after a spawn event."""
        self.grad_norms = []
        self.domain_losses = {'code': [], 'medical': []}

print("ConflictSaturationMonitor defined.")

In [ ]:
# ── Data preparation helpers ──────────────────────────────────────────────────
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

def load_split_auto(dataset_name, config_name=None, preferred="train"):
    """Load dataset, auto-selecting split."""
    kwargs = {"name": config_name} if config_name else {}
    ds_dict = load_dataset(dataset_name, **kwargs)
    available = list(ds_dict.keys())
    split = preferred if preferred in available else ("test" if "test" in available else available[0])
    print(f"    [{dataset_name}] using split='{split}' (available: {available})")
    return ds_dict[split]

def build_calibration_dataloader(tokenizer, conflict_ratio=0.2, max_length=128, n_each=300):
    """
    Build interleaved code/medical dataset for calibration.
    """
    print("Loading datasets...")
    code_ds = load_split_auto("openai/openai_humaneval", None, "test")
    med_ds = load_split_auto("qiaojin/PubMedQA", "pqa_labeled", "train")
    
    code_texts = [ex["prompt"] for ex in code_ds.select(range(min(n_each, len(code_ds))))]
    med_texts = [ex["question"] for ex in med_ds.select(range(min(n_each, len(med_ds))))]
    
    # Interleave based on conflict ratio
    texts = []
    domains = []
    
    n_code = int(n_each * (1 - conflict_ratio))
    n_med = int(n_each * conflict_ratio)
    
    for i in range(max(n_code, n_med)):
        if i < n_code:
            texts.append(code_texts[i % len(code_texts)])
            domains.append('code')
        if i < n_med:
            texts.append(med_texts[i % len(med_texts)])
            domains.append('medical')
    
    # Tokenize
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_tensors='pt'
    )
    
    print(f"Built dataset: {len(texts)} samples ({n_code} code, {n_med} medical)")
    return encodings, domains

print("Data helpers defined.")

In [ ]:
# ── HumanEval pass@1 Evaluation ───────────────────────────────────────────────
import subprocess
import tempfile

def run_in_sandbox(code, timeout=5):
    """Execute code in sandbox, return (success, error_msg)."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        fname = f.name
    try:
        r = subprocess.run(["python3", fname], capture_output=True, timeout=timeout)
        return r.returncode == 0, r.stderr.decode()[:200] if r.stderr else ""
    except subprocess.TimeoutExpired:
        return False, "TIMEOUT"
    except Exception as e:
        return False, str(e)[:200]
    finally:
        os.unlink(fname)

def evaluate_humaneval(model, tokenizer, max_problems=164, max_new_tokens=256):
    """
    Evaluate model on HumanEval benchmark.
    Returns pass@1 score.
    """
    humaneval = load_dataset("openai/openai_humaneval", split="test")
    device = next(model.parameters()).device
    
    model.eval()
    passed = 0
    
    for i, prob in enumerate(humaneval):
        if i >= max_problems:
            break
        
        # Qwen chat format
        prompt = f"Complete this Python function. Return only code, no explanation:\n\n{prob['prompt']}"
        
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Extract code from completion (handle markdown code blocks)
        if "```python" in completion:
            completion = completion.split("```python")[1].split("```")[0]
        elif "```" in completion:
            completion = completion.split("```")[1].split("```")[0]
        
        full_code = prob["prompt"] + completion + "\n" + prob["test"] + f"\ncheck({prob['entry_point']})"
        
        success, err = run_in_sandbox(full_code)
        if success:
            passed += 1
        
        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{min(max_problems, len(humaneval))}] pass@1: {passed/(i+1):.3f}")
    
    total = min(max_problems, len(humaneval))
    p1 = passed / total
    print(f"\nFinal pass@1: {p1:.4f} ({passed}/{total})")
    return p1

print("HumanEval evaluation defined.")

In [ ]:
# ── Quick HumanEval test (5 problems) to verify everything works ─────────────
print("Running quick HumanEval test (5 problems)...")
print("="*60)

# Make sure model is in original state
layer0 = model.model.layers[0]
if hasattr(layer0.mlp, 'original'):
    layer0.mlp = layer0.mlp.original

p1 = evaluate_humaneval(model, tokenizer, max_problems=5)
print(f"\nQuick test pass@1: {p1:.2%}")

---

## Next Steps

If the model load test and quick HumanEval test pass, you can proceed to:

1. **Phase 1**: Apply wrapper and train base LoRA adapters
2. **Phase 2**: Calibrate spawning thresholds
3. **Phase 3**: Full training with hierarchical spawning
4. **Evaluation**: Full HumanEval pass@1 comparison

The cells below are templates that need to be filled in based on the architecture exploration results.